# Raster Preprocessing & Patch Extraction PipelineThis notebook prepares geospatial covariates for the CNN-based density ratio estimation models.## Pipeline Overview```Raw GeoTIFFs ──► RasterPreprocessor ──► Normalised GeoTIFFs                                              │CSV (coords, labels, folds) ──────────────────┤                                              ▼                                    RasterPatchExtractor                                              │                                              ▼                                   X.npy  (N, C, H, W)                                   M.npy  (N, C, H, W)  masks                                   y.npy  (N,)           labels                                   fold.npy (N,)         CV folds```**Step 1 – Preprocessing**: Each raster is loaded, cleaned (nodata → NaN, absurd values removed), optionally log-transformed, and normalised (median/IQR by default).**Step 2 – Patch extraction**: For each point in the CSV, a `patch_size × patch_size` window is extracted from every covariate raster. Missing pixels are median-imputed and a binary mask records which pixels were observed.

## 1. Imports

In [ ]:
import os
from pathlib import Path
from typing import Optional, Literal, List, Tuple

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.plot import show
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import matplotlib.pyplot as plt

## 2. ConfigurationUpdate these paths to match your directory structure.

In [ ]:
# ── Input rasters ─────────────────────────────────────────
RAW_RASTER_DIR       = "data/covariates/raw"
PROCESSED_RASTER_DIR = "data/covariates/processed"

# ── CSV splits ────────────────────────────────────────────
CV_CSV_PATH   = "data/splits/cv_split.csv"
TEST_CSV_PATH = "data/splits/test_split.csv"

# ── Output patches ────────────────────────────────────────
PATCH_OUTPUT_BASE = "data/patches"

# ── Preprocessing defaults ────────────────────────────────
NORMALIZATION = "median_iqr"   # "minmax" | "zscore" | "median_iqr" | None
LOG_TRANSFORM = False          # set True only if rasters are NOT already log-transformed
DST_NODATA    = -9999.0        # sentinel value written to disk for missing pixels

# ── Patch extraction defaults ─────────────────────────────
PATCH_SIZES   = [13]           # list of patch sizes to extract (must be odd)
NUM_WORKERS   = 12             # threads for parallel extraction
OVERWRITE     = True           # re-extract even if outputs exist

## 3. Raster PreprocessorLoads a single-band GeoTIFF, converts nodata to NaN, optionally applies a log transform, and normalises the values.Supported normalisations:- **minmax**: scales to \[0, 1\] using global min/max  - **zscore**: centres on mean, scales by std  - **median_iqr**: centres on median, scales by interquartile range (robust to outliers)

In [ ]:
class RasterPreprocessor:
    """Load, clean, and normalise a single-band raster."""

    def __init__(
        self,
        raster_path: str,
        band: int = 1,
        normalization: Optional[Literal["minmax", "zscore", "median_iqr"]] = "median_iqr",
        log_transform: bool = False,
        offset: float = 1e-6,
    ):
        if not os.path.exists(raster_path):
            raise FileNotFoundError(f"Raster not found: {raster_path}")

        self.raster_path = raster_path
        self.band = band
        self.normalization = normalization
        self.log_transform = log_transform
        self.offset = offset

        with rasterio.open(raster_path) as src:
            if band < 1 or band > src.count:
                raise ValueError(f"Band {band} unavailable; raster has {src.count} band(s).")
            self.profile = src.profile.copy()

        self.array = self._load_and_normalize()

    # ── core logic ────────────────────────────────────────

    def _load_and_normalize(self) -> np.ndarray:
        with rasterio.open(self.raster_path) as src:
            arr = src.read(self.band, out_dtype="float32")
            nodata = src.nodata

        # nodata → NaN
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        # guard extreme values
        arr[(arr < -1e16) | (arr > 1e16)] = np.nan

        # optional log (only if raster is NOT already log-transformed)
        if self.log_transform:
            pos = arr > 0
            if np.any(pos):
                arr[pos] = np.log(arr[pos] + self.offset)

        if self.normalization is None:
            return arr.astype("float32")

        v = arr[np.isfinite(arr)]
        if v.size == 0:
            return arr.astype("float32")

        eps = 1e-8

        if self.normalization == "minmax":
            vmin, vmax = float(v.min()), float(v.max())
            rng = vmax - vmin
            if rng < eps:
                return arr.astype("float32")
            arr = (arr - vmin) / rng

        elif self.normalization == "zscore":
            mean, std = float(v.mean()), float(v.std())
            if std < eps:
                return arr.astype("float32")
            arr = (arr - mean) / std

        elif self.normalization == "median_iqr":
            med = float(np.median(v))
            q1, q3 = float(np.percentile(v, 25)), float(np.percentile(v, 75))
            iqr = q3 - q1
            if iqr < eps:
                return arr.astype("float32")
            arr = (arr - med) / iqr

        else:
            raise ValueError(f"Unknown normalization: {self.normalization}")

        return arr.astype("float32")

    # ── I/O helpers ───────────────────────────────────────

    def save_array(self, output_path: str, array: np.ndarray, dtype: str = "float32"):
        """Write a 2-D array back to a single-band GeoTIFF."""
        prof = self.profile.copy()
        prof.update(dtype=dtype, nodata=DST_NODATA, count=1)
        with rasterio.open(output_path, "w", **prof) as dst:
            out = np.where(np.isfinite(array), array, DST_NODATA).astype(dtype)
            dst.write(out, 1)

    def plot_raster(
        self,
        cmap: str = "viridis",
        title: Optional[str] = None,
        colorbar: bool = True,
        figsize: Tuple[int, int] = (10, 8),
    ):
        plt.figure(figsize=figsize)
        ax = plt.gca()
        show(self.array, transform=self.profile["transform"], cmap=cmap, ax=ax)
        if colorbar:
            plt.colorbar(label="Value", shrink=0.7)
        plt.title(title or f"{os.path.basename(self.raster_path)} (Band {self.band})")
        plt.tight_layout()
        plt.show()

### Batch preprocessing utility

In [ ]:
def preprocess_rasters(
    input_dir: str,
    output_dir: str,
    band: int = 1,
    pattern: str = "*.tif",
    overwrite: bool = False,
    normalization: Optional[Literal["minmax", "zscore", "median_iqr"]] = "median_iqr",
    log_transform: bool = False,
) -> List[str]:
    """Preprocess all rasters matching `pattern` and write to `output_dir`."""
    in_dir  = Path(input_dir)
    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_paths: List[str] = []
    for p in sorted(in_dir.glob(pattern)):
        out_path = out_dir / f"processed_{p.name}"
        if out_path.exists() and not overwrite:
            out_paths.append(str(out_path))
            continue

        rp = RasterPreprocessor(
            raster_path=str(p),
            band=band,
            normalization=normalization,
            log_transform=log_transform,
        )
        rp.save_array(str(out_path), rp.array)
        out_paths.append(str(out_path))

    print(f"Preprocessed {len(out_paths)} raster(s) → {output_dir}")
    return out_paths

## 4. Raster Patch ExtractorExtracts `patch_size × patch_size` windows from every covariate raster at each point location.Key design decisions:- **Thread-safe**: each worker opens its own GDAL handle — no shared file pointers  - **Median imputation**: NaN pixels are filled with the raster-wide median (computed once on a downsampled thumbnail)  - **Binary mask**: records which pixels were genuinely observed vs. imputed

In [ ]:
class RasterPatchExtractor:
    """
    Extract multi-channel patches from a directory of preprocessed rasters.

    Returns arrays shaped [N, C, H, W] where:
        N = number of spatial points
        C = number of covariate rasters
        H = W = patch_size  (must be odd)
    """

    def __init__(self, raster_dir: str, patch_size: int = 13, pattern: str = "*.tif"):
        if patch_size % 2 == 0:
            raise ValueError(f"patch_size must be odd, got {patch_size}")
        self.patch_size = patch_size
        self.half_size  = patch_size // 2

        self.raster_paths = sorted(Path(raster_dir).glob(pattern))
        if not self.raster_paths:
            raise FileNotFoundError(f"No rasters in {raster_dir} matching {pattern}")

        self.raster_names = [p.stem for p in self.raster_paths]
        self._cached_medians: Optional[List[float]] = None

    # ── single-patch extraction ───────────────────────────

    def _extract_patch(self, raster, x: float, y: float) -> np.ndarray:
        """
        Extract a single patch centred on (x, y) in the raster's CRS.
        Returns (patch_size, patch_size) float32 with NaN for missing pixels.
        """
        col_c, row_c = ~raster.transform * (x, y)
        row_c, col_c = int(round(row_c)), int(round(col_c))

        # full window [start, end)
        rs, re = row_c - self.half_size, row_c + self.half_size + 1
        cs, ce = col_c - self.half_size, col_c + self.half_size + 1

        # clip to raster bounds
        rs_c, cs_c = max(0, rs), max(0, cs)
        re_c, ce_c = min(raster.height, re), min(raster.width, ce)
        h, w = re_c - rs_c, ce_c - cs_c

        if h <= 0 or w <= 0:
            return np.full((self.patch_size, self.patch_size), np.nan, dtype="float32")

        patch = raster.read(1, window=Window(cs_c, rs_c, w, h), boundless=False).astype("float32")

        if raster.nodata is not None:
            patch = np.where(patch == raster.nodata, np.nan, patch)

        # pad back to full patch_size
        pad_top    = rs_c - rs
        pad_left   = cs_c - cs
        pad_bottom = self.patch_size - pad_top  - patch.shape[0]
        pad_right  = self.patch_size - pad_left - patch.shape[1]

        return np.pad(
            patch,
            ((pad_top, pad_bottom), (pad_left, pad_right)),
            mode="constant", constant_values=np.nan,
        ).astype("float32")

    # ── raster medians (for imputation) ───────────────────

    def _compute_raster_medians(self, downscale: int = 8) -> List[float]:
        """Compute medians on downsampled thumbnails for speed."""
        meds = []
        for p in self.raster_paths:
            with rasterio.open(p) as r:
                out_h = max(1, r.height // downscale)
                out_w = max(1, r.width  // downscale)
                thumb = r.read(1, out_shape=(1, out_h, out_w),
                               resampling=Resampling.nearest).astype("float32")
                if r.nodata is not None:
                    thumb = np.where(thumb == r.nodata, np.nan, thumb)
                meds.append(float(np.nanmedian(thumb)))
        return meds

    # ── main extraction entry point ───────────────────────

    def extract_patches_from_csv(
        self,
        csv_path,
        save_to_disk: bool = False,
        output_dir: str = "patches",
        num_workers: int = 12,
    ) -> Tuple[np.ndarray, np.ndarray, List[str]]:
        """
        Extract patches for all points in a CSV (or DataFrame).

        Returns:
            data:  (N, C, H, W) float32 — imputed patch values
            mask:  (N, C, H, W) float32 — 1 = observed, 0 = imputed
            names: list of C raster channel names
        """
        df = pd.read_csv(csv_path) if isinstance(csv_path, str) else csv_path
        coords = df[["Longitude", "Latitude"]].drop_duplicates().to_numpy(dtype="float64")
        xs, ys = coords[:, 0], coords[:, 1]

        N, C = len(xs), len(self.raster_paths)
        PS = self.patch_size
        data_arr = np.full((N, PS, PS, C), np.nan, dtype=np.float32)
        mask_arr = np.zeros_like(data_arr)

        if self._cached_medians is None:
            self._cached_medians = self._compute_raster_medians()
        medians = self._cached_medians

        for ridx, rpath in enumerate(tqdm(self.raster_paths, desc="Extracting patches")):
            med = medians[ridx]

            def worker(i):
                try:
                    with rasterio.open(rpath) as r:
                        patch = self._extract_patch(r, xs[i], ys[i])
                except Exception:
                    return i, None, None
                m = np.isfinite(patch)
                filled = np.where(m, patch, med).astype("float32")
                return i, filled, m.astype("float32")

            with ThreadPoolExecutor(max_workers=num_workers) as ex:
                for i, filled, m in ex.map(worker, range(N)):
                    if filled is not None:
                        data_arr[i, :, :, ridx] = filled
                        mask_arr[i, :, :, ridx] = m

        # transpose to channel-first: (N, C, H, W)
        data_out = np.transpose(data_arr, (0, 3, 1, 2))
        mask_out = np.transpose(mask_arr, (0, 3, 1, 2))

        if save_to_disk:
            out = Path(output_dir)
            out.mkdir(parents=True, exist_ok=True)
            np.save(out / "all_patches.npy", data_out)
            np.save(out / "all_masks.npy",   mask_out)
            np.save(out / "raster_names.npy", np.array(self.raster_names, dtype=object))
            print(f"Saved patches to {output_dir}")

        return data_out, mask_out, self.raster_names

    def close(self):
        """No-op (kept for API compatibility)."""
        pass

## 5. Patch Extraction with MetadataWraps the extractor to produce the full set of arrays expected by the DRE training pipeline: `X.npy`, `M.npy`, `y.npy`, `fold.npy`, and associated metadata.

In [ ]:
def extract_patches_with_metadata(
    extractor: RasterPatchExtractor,
    csv_path: str,
    lon_col: str = "Longitude",
    lat_col: str = "Latitude",
    label_col: str = "label",
    fold_col: str = "fold",
    cluster_col: str = "cluster_id",
    save_dir: str = "dre_data",
    overwrite: bool = False,
):
    """
    Extract patches aligned to CSV rows and save training-ready arrays.

    Saves to `save_dir`:
        X.npy            (N, C, H, W)  covariate patches
        M.npy            (N, C, H, W)  observation masks
        y.npy            (N,)          binary labels
        fold.npy         (N,)          CV fold assignments
        cluster.npy      (N,)          spatial cluster IDs
        patch_names.npy  (C,)          covariate names
        meta.parquet                   original CSV for traceability
    """
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    expected = ["X.npy", "M.npy", "y.npy", "fold.npy", "patch_names.npy"]
    if not overwrite and all((save_dir / f).exists() for f in expected):
        print(f"[skip] Files already exist in {save_dir}")
        return

    # ── load CSV ──────────────────────────────────────────
    df = pd.read_csv(csv_path)
    for col in [lon_col, lat_col, label_col]:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # ── labels → 0/1 ─────────────────────────────────────
    y = (
        df[label_col].astype(str).str.lower().str.startswith("pres")
        .astype(np.float32).values
    )

    # ── fold & cluster (may be absent or NaN) ─────────────
    fold = df[fold_col].values if fold_col in df.columns else np.full(len(df), np.nan)
    cluster = (
        df[cluster_col].fillna(-1).astype(np.int32).values
        if cluster_col in df.columns
        else np.full(len(df), -1, dtype=np.int32)
    )

    # ── deduplicate coordinates for extraction ────────────
    key = df[[lon_col, lat_col]].apply(tuple, axis=1)
    _, inverse = np.unique(key, return_inverse=True)
    uniq_df = df[[lon_col, lat_col]].drop_duplicates()

    if lon_col != "Longitude" or lat_col != "Latitude":
        uniq_df = uniq_df.rename(columns={lon_col: "Longitude", lat_col: "Latitude"})

    Xu, Mu, patch_names = extractor.extract_patches_from_csv(
        uniq_df, save_to_disk=False, num_workers=NUM_WORKERS,
    )

    # ── map back to full CSV row order ────────────────────
    X = Xu[inverse]
    M = Mu[inverse]

    # ── save ──────────────────────────────────────────────
    np.save(save_dir / "X.npy",           X.astype(np.float32))
    np.save(save_dir / "M.npy",           M.astype(np.float32))
    np.save(save_dir / "y.npy",           y)
    np.save(save_dir / "fold.npy",        fold)
    np.save(save_dir / "cluster.npy",     cluster)
    np.save(save_dir / "patch_names.npy", np.array(patch_names, dtype=object))
    df.to_parquet(save_dir / "meta.parquet", index=False)

    print(f"[ok] Saved to {save_dir}")
    print(f"     X {X.shape}  M {M.shape}  y {y.shape}  channels {len(patch_names)}")

---## 6. Run: Preprocess RastersNormalise all raw covariate rasters and write the results to `PROCESSED_RASTER_DIR`.

In [ ]:
processed_paths = preprocess_rasters(
    input_dir=RAW_RASTER_DIR,
    output_dir=PROCESSED_RASTER_DIR,
    normalization=NORMALIZATION,
    log_transform=LOG_TRANSFORM,
    overwrite=False,
)
print(f"{len(processed_paths)} raster(s) preprocessed.")

## 7. Run: Extract PatchesExtract covariate patches for both the CV and test splits at each requested `patch_size`.

In [ ]:
for ps in PATCH_SIZES:
    print(f"\n{'='*60}")
    print(f"  Patch size = {ps}")
    print(f"{'='*60}")

    extractor = RasterPatchExtractor(
        raster_dir=PROCESSED_RASTER_DIR,
        patch_size=ps,
        pattern="*.tif",
    )

    # ── Test split ────────────────────────────────────────
    print(f"\n--- Test split ---")
    extract_patches_with_metadata(
        extractor=extractor,
        csv_path=TEST_CSV_PATH,
        save_dir=os.path.join(PATCH_OUTPUT_BASE, f"test_patches_{ps}"),
        overwrite=OVERWRITE,
    )

    # ── CV split ──────────────────────────────────────────
    print(f"\n--- CV split ---")
    extract_patches_with_metadata(
        extractor=extractor,
        csv_path=CV_CSV_PATH,
        save_dir=os.path.join(PATCH_OUTPUT_BASE, f"cv_patches_{ps}"),
        overwrite=OVERWRITE,
    )

    extractor.close()

print("\nDone. Patches are ready for model training.")

## 8. (Optional) Inspect Extracted Patches

In [ ]:
# Quick sanity check on the CV patches
ps = PATCH_SIZES[0]
cv_dir = Path(PATCH_OUTPUT_BASE) / f"cv_patches_{ps}"

if (cv_dir / "X.npy").exists():
    X = np.load(cv_dir / "X.npy")
    M = np.load(cv_dir / "M.npy")
    y = np.load(cv_dir / "y.npy")
    fold = np.load(cv_dir / "fold.npy")
    names = np.load(cv_dir / "patch_names.npy", allow_pickle=True)

    print(f"X shape:     {X.shape}  (N, C, H, W)")
    print(f"M shape:     {M.shape}")
    print(f"y shape:     {y.shape}  |  class balance: {y.mean():.4f}")
    print(f"Folds:       {np.unique(fold[np.isfinite(fold)]).astype(int)}")
    print(f"Channels:    {list(names)}")
    print(f"Mask coverage: {M.mean():.2%} pixels observed")

    # show one example patch per channel
    idx = 0
    fig, axes = plt.subplots(1, min(len(names), 6), figsize=(18, 3))
    if not hasattr(axes, '__len__'):
        axes = [axes]
    for c, ax in enumerate(axes):
        ax.imshow(X[idx, c], cmap="viridis")
        ax.set_title(names[c][:15], fontsize=8)
        ax.axis("off")
    plt.suptitle(f"Sample patch (index {idx}, label={int(y[idx])})", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print(f"No patches found at {cv_dir}. Run extraction first.")